In [1]:
data_path = '/data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract_and_FC/ECtrained>training/FCtraining_outLinearSigmaOnly/*.bp'
save_path = '2trainBPTT_Deco_EndToEnd_ftract_and_FC_ECtrained>training-trainingFC_SigmaOnly.npz'
num_parallel = 50

In [2]:
import os,sys
import brainpy as bp
import brainpy.math as bm
print('bp version: ', bp.__version__)
bm.set_platform('cpu')
bm.set_host_device_count(num_parallel)

import matplotlib.pyplot as plt
import numpy as np
from glob import glob

sys.path.append("/data/xyc/codes/iEEG_network/3_analysis_EC")
sys.path.append("/data/xyc/codes/iEEG_network/4_model_BrainPy")
from func_fetch_conn import fetch_conn_ftract,fetch_conn_enigma,fetch_conn_Kong2021
from func_simulation import def_bodyEC,def_bodyFC,get_parallel_parameters_from_bppaths

bp version:  2.6.0


In [3]:
# loading SC
ctx_labels,sc_ctx, fc_ctx = fetch_conn_enigma() # sc is logged, fc is z-transformed
# ctx_labels,sc_ctx, fc_ctx = fetch_conn_Kong2021() # fc is R-value

# loading EC
# ctx_labels,ftract_ctx,ftract_ctx_amplitud,ftract_ctx_integral,ftract_ctx_duration = fetch_conn_ftract()

# num of nodes
N = len(ctx_labels)

# 除以最大值标准化
struc_conn_matrix = sc_ctx / sc_ctx.max()

In [4]:
bm.dt = 5e-4 # second; 2000Hz sample rate
bm.set_mode(bm.batching_mode)

Load paths

In [5]:
bppaths = np.asarray(glob(data_path))
print(bppaths.size)

925


In [6]:
Gs,ws,Is,sigmas,Ss=get_parallel_parameters_from_bppaths(bppaths)

Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract_and_FC/ECtrained>training/FCtraining_outLinearSigmaOnly/FC<lr0.01>EC<G1.0_w-111.0_I0.0_Adam_lr0.05_wLRx10_epo10k_maxNorm>.bp
0/925
Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract_and_FC/ECtrained>training/FCtraining_outLinearSigmaOnly/FC<lr0.01>EC<G1.0_w-111.0_I0.0_Adam_lr0.05_wLRx10_epo10k_maxNorm>.bp
1/925
Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract_and_FC/ECtrained>training/FCtraining_outLinearSigmaOnly/FC<lr0.05>EC<G1.0_w-101.0_I0.0_Adam_lr0.1_wLRx1_epo10k_outLinear>.bp
2/925
Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract_and_FC/ECtrained>training/FCtraining_outLinearSigmaOnly/FC<lr0.01>EC<G1.0_w-70.0_I0.0_Adam_lr0.1_wLRx1_epo10k_outLinear>.bp
3/925
Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2tra

In [7]:
Ss=Ss.mean(axis=1) # batch size = 1 for FC simulation, averaging across batch axis
print(Ss.shape)
print(np.isnan(Ss.mean(axis=(1,))).sum())
print(np.isnan(sigmas.mean(axis=(1,))).sum())

(925, 68)
218
218


RUN outIdentity

In [8]:
# resting-state with noise input
rng = bm.random.RandomState(123)
warmupinput = rng.randn(1,int(2/bm.dt),N)* bm.sqrt(bm.dt) # 2 seconds
restinput = rng.randn(1,int(2/bm.dt),N)* bm.sqrt(bm.dt) # 2 seconds

bodyFC_outIdentity = def_bodyFC(struc_conn_matrix,restinput,warmupinput,ifoutBalloon=False)

def bodyFC_parallel(G,w,I,sigma,S=0.0):
    _,FC=bodyFC_outIdentity(G,w,I,sigma,S)
    return FC

In [9]:
%%time
FCs_outIdentity = bp.running.jax_parallelize_map(func=bodyFC_parallel,num_parallel=num_parallel,clear_buffer=False,arguments=[Gs,ws,Is,sigmas,Ss]) 
print(FCs_outIdentity.shape)

(925, 68, 68)
CPU times: user 16.4 s, sys: 5.11 s, total: 21.5 s
Wall time: 1.07 s


RUN outBalloon

In [10]:
# resting-state with noise input
rng = bm.random.RandomState(123)
warmupinput = rng.randn(1,int(5*60/bm.dt),N)* bm.sqrt(bm.dt) # 5 min (5*60 seconds) for balloon(S)
restinput = rng.randn(1,int(10*60/bm.dt),N)* bm.sqrt(bm.dt) # 10 min (10*60 seconds) for balloon(S)

bodyFC_outBalloon = def_bodyFC(struc_conn_matrix,restinput,warmupinput,ifoutBalloon=True)

def bodyFC_parallel(G,w,I,sigma,S=0.0):
    _,FC=bodyFC_outBalloon(G,w,I,sigma,S)
    return FC

In [12]:
%%time
FCs_outBalloon = bp.running.jax_parallelize_map(func=bodyFC_parallel,num_parallel=num_parallel,clear_buffer=False,arguments=[Gs,ws,Is,sigmas,Ss]) 
print(FCs_outBalloon.shape)

(925, 68, 68)
CPU times: user 1h 52min 46s, sys: 15min 52s, total: 2h 8min 39s
Wall time: 2min 12s


save

In [13]:
#save
np.savez(save_path, bppaths=np.asarray(bppaths), FCs_outIdentity=np.asarray(FCs_outIdentity), FCs_outBalloon=np.asarray(FCs_outBalloon), )